# Sephora Product Intelligence — Data Cleaning

## Purpose
This notebook executes all cleaning and filtering decisions made during EDA.
The output is a clean, analysis-ready dataset saved to `data/processed/` 
that will be used as the foundation for all feature engineering.

## What this notebook does
1. Reloads raw data
2. Drops irrelevant columns
3. Fixes data types and casing
4. Filters to chosen categories
5. Handles missing values
6. Validates the cleaned output
7. Saves to `data/processed/`

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries loaded")

Libraries loaded


## 2. Reload Raw Data
Always reload from raw in every notebook — never depend on variables 
from another notebook. This makes each notebook self-contained and reproducible.

In [2]:
# Load products
products = pd.read_csv("../data/raw/product_info.csv")

# Load all review files
review_files = glob.glob("../data/raw/reviews_*.csv")
reviews = pd.concat([pd.read_csv(f) for f in review_files], ignore_index=True)

print(f"Raw products: {products.shape}")
print(f"Raw reviews: {reviews.shape}")

Raw products: (8494, 27)
Raw reviews: (1094411, 19)


## 3. Drop Irrelevant Columns
Removing columns that are either mostly empty, not useful for analysis,
or redundant with other columns we already have.

In [3]:
# Columns to drop from products
drop_product_cols = [
    "sale_price_usd",    # 96.8% null
    "value_price_usd",   # 94.7% null
    "variation_desc",    # 85.3% null, not useful for analysis
    "child_max_price",   # 67.6% null, redundant with price_usd
    "child_min_price",   # 67.6% null, redundant with price_usd
]

products = products.drop(columns=drop_product_cols)

# Columns to drop from reviews
drop_review_cols = [
    "Unnamed: 0",        # junk index column
]

reviews = reviews.drop(columns=drop_review_cols)

print(f"Products shape after dropping columns: {products.shape}")
print(f"Reviews shape after dropping columns: {reviews.shape}")
print(f"\nProduct columns remaining: {list(products.columns)}")

Products shape after dropping columns: (8494, 22)
Reviews shape after dropping columns: (1094411, 18)

Product columns remaining: ['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value', 'ingredients', 'price_usd', 'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'child_count']


## 4. Fix Data Types & Casing
Standardizing text fields so that "CLINIQUE", "Clinique", and "clinique" 
are all treated as the same brand. Also fixing date types for reviews.

In [4]:
# Fix casing on products
products["brand_name"] = products["brand_name"].str.strip().str.title()
products["product_name"] = products["product_name"].str.strip()
products["primary_category"] = products["primary_category"].str.strip().str.title()
products["secondary_category"] = products["secondary_category"].str.strip().str.title()
products["tertiary_category"] = products["tertiary_category"].str.strip().str.title()

# Fix casing on reviews
reviews["brand_name"] = reviews["brand_name"].str.strip().str.title()
reviews["skin_type"] = reviews["skin_type"].str.strip().str.lower()
reviews["skin_tone"] = reviews["skin_tone"].str.strip().str.lower()

# Fix date type on reviews
reviews["submission_time"] = pd.to_datetime(reviews["submission_time"])

# Verify
print("=== SAMPLE BRAND NAMES AFTER CASING FIX ===")
print(products["brand_name"].value_counts().head(10))
print(f"\nReview date range: {reviews['submission_time'].min()} to {reviews['submission_time'].max()}")

=== SAMPLE BRAND NAMES AFTER CASING FIX ===
brand_name
Sephora Collection         352
Clinique                   179
Dior                       136
Tarte                      131
Nest New York              115
Bumble And Bumble          110
Kérastase                  108
Tom Ford                   100
Charlotte Tilbury           99
Anastasia Beverly Hills     95
Name: count, dtype: int64

Review date range: 2008-08-28 00:00:00 to 2023-03-21 00:00:00


## 5. Filter to Chosen Categories
Applying the scope decision made in EDA — keeping only Skincare, Makeup, 
and Hair products in our chosen secondary categories.

In [9]:
keep_skincare = [
    "Moisturizers", "Treatments", "Cleansers", "Eye Care",
    "Masks", "Sunscreen", "Lip Balms & Treatments", "Self Tanners"
]

products_filtered = products[
    (products["primary_category"] == "Skincare") & 
    (products["secondary_category"].isin(keep_skincare))
].copy()

print(f"Products before filter: {len(products)}")
print(f"Products after filter: {len(products_filtered)}")
print(f"\nBreakdown by secondary category:")
print(products_filtered["secondary_category"].value_counts())

Products before filter: 8494
Products after filter: 1952

Breakdown by secondary category:
secondary_category
Moisturizers              551
Treatments                466
Cleansers                 361
Eye Care                  186
Masks                     166
Sunscreen                 108
Lip Balms & Treatments     61
Self Tanners               53
Name: count, dtype: int64


## 6. Filter Reviews to Match Products
We only want reviews for products we kept. We filter reviews by 
matching on product_id so both tables stay in sync.

In [10]:
# Get the set of valid product IDs after filtering
valid_product_ids = set(products_filtered["product_id"])

# Filter reviews to only include our kept products
reviews_filtered = reviews[reviews["product_id"].isin(valid_product_ids)].copy()

print(f"Reviews before filter: {len(reviews)}")
print(f"Reviews after filter: {len(reviews_filtered)}")
print(f"Reviews removed: {len(reviews) - len(reviews_filtered)}")
print(f"\nUnique products with reviews: {reviews_filtered['product_id'].nunique()}")
print(f"Products with no reviews: {len(valid_product_ids) - reviews_filtered['product_id'].nunique()}")

Reviews before filter: 1094411
Reviews after filter: 980344
Reviews removed: 114067

Unique products with reviews: 1915
Products with no reviews: 37


## 7. Handle Missing Values

### Key finding from filter step:
- We are focusing exclusively on Skincare products where review data is complete
- Makeup and Hair were explored in EDA but excluded from scoring due to missing review data
- All 1,952 skincare products have potential review coverage
- Products with no reviews will be kept in the dataset but flagged — 
  they won't receive a worth-the-hype score

In [11]:
# Products — handle missing values

# Drop products with no price (can't do price analysis without it)
print(f"Products with no price: {products_filtered['price_usd'].isnull().sum()}")
print(f"Products with no rating: {products_filtered['rating'].isnull().sum()}")

# Fill boolean flags with 0 (assume not limited/new/online if not specified)
bool_cols = ["limited_edition", "new", "online_only", "out_of_stock", "sephora_exclusive"]
products_filtered[bool_cols] = products_filtered[bool_cols].fillna(0).astype(int)

# Drop rows with no price — can't analyze without it
products_clean = products_filtered.dropna(subset=["price_usd"]).copy()

print(f"\nProducts after dropping no-price rows: {len(products_clean)}")
print(f"Products with no rating (kept but flagged): {products_clean['rating'].isnull().sum()}")

Products with no price: 0
Products with no rating: 37

Products after dropping no-price rows: 1952
Products with no rating (kept but flagged): 37


In [12]:
# Reviews — handle missing values

# Drop reviews with no rating (core metric, can't impute)
reviews_clean = reviews_filtered.dropna(subset=["rating"]).copy()

# Fill helpfulness nulls with 0 (no votes = no helpfulness signal)
reviews_clean["helpfulness"] = reviews_clean["helpfulness"].fillna(0)

# Fill is_recommended nulls with median
reviews_clean["is_recommended"] = reviews_clean["is_recommended"].fillna(
    reviews_clean["is_recommended"].median()
)

# Add review year for time-based analysis later
reviews_clean["review_year"] = reviews_clean["submission_time"].dt.year

print(f"Reviews before null handling: {len(reviews_filtered)}")
print(f"Reviews after null handling: {len(reviews_clean)}")
print(f"\nNull check after cleaning:")
print(reviews_clean[["rating", "helpfulness", "is_recommended"]].isnull().sum())

Reviews before null handling: 980344
Reviews after null handling: 980344

Null check after cleaning:
rating            0
helpfulness       0
is_recommended    0
dtype: int64


## 8. Final Validation
One last check before saving — confirming the cleaned dataset is consistent, 
complete, and ready for feature engineering.

In [13]:
print("=== FINAL VALIDATION ===")
print(f"\nProducts:")
print(f"  Total skincare products: {len(products_clean)}")
print(f"  Price range: ${products_clean['price_usd'].min():.2f} - ${products_clean['price_usd'].max():.2f}")
print(f"  Avg price: ${products_clean['price_usd'].mean():.2f}")
print(f"  Products with rating: {products_clean['rating'].notna().sum()}")
print(f"  Products without rating (flagged): {products_clean['rating'].isna().sum()}")

print(f"\nReviews:")
print(f"  Total reviews: {len(reviews_clean)}")
print(f"  Unique products reviewed: {reviews_clean['product_id'].nunique()}")
print(f"  Rating distribution:")
print(reviews_clean["rating"].value_counts().sort_index())
print(f"  Date range: {reviews_clean['submission_time'].min().date()} to {reviews_clean['submission_time'].max().date()}")

print(f"\nConsistency check:")
print(f"  All review product_ids exist in products: {reviews_clean['product_id'].isin(products_clean['product_id']).all()}")
print(f"  Duplicate products: {products_clean.duplicated(subset=['product_id']).sum()}")
print(f"  Duplicate reviews: {reviews_clean.duplicated().sum()}")

=== FINAL VALIDATION ===

Products:
  Total skincare products: 1952
  Price range: $3.00 - $425.00
  Avg price: $58.04
  Products with rating: 1915
  Products without rating (flagged): 37

Reviews:
  Total reviews: 980344
  Unique products reviewed: 1915
  Rating distribution:
rating
1     54101
2     47252
3     72668
4    179963
5    626360
Name: count, dtype: int64
  Date range: 2008-08-28 to 2023-03-21

Consistency check:
  All review product_ids exist in products: True
  Duplicate products: 0
  Duplicate reviews: 214


## 9. Save Cleaned Data


In [14]:
import os

# Create processed folder if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save cleaned products
products_clean.to_csv("../data/processed/products_clean.csv", index=False)

# Save cleaned reviews
reviews_clean.to_csv("../data/processed/reviews_clean.csv", index=False)

print("=== FILES SAVED ===")
print(f"products_clean.csv — {len(products_clean)} rows, {products_clean.shape[1]} columns")
print(f"reviews_clean.csv — {len(reviews_clean)} rows, {reviews_clean.shape[1]} columns")
print(f"\nSaved to: data/processed/")

=== FILES SAVED ===
products_clean.csv — 1952 rows, 22 columns
reviews_clean.csv — 980344 rows, 19 columns

Saved to: data/processed/
